# 05 ML GNN Embeddings

This notebook uses `MLTrainAndStore` to train regressors with only the GraphSAGE embedding features.

In [1]:
from pathlib import Path
import sys
import os

sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd

from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
import xgboost as xgb
from xgboost import XGBRegressor

from src.models.ml_train_and_store import (
    MLTrainAndStore,
    load_gnn_ml_dataset,
    make_log_regression_model,
)

pd.set_option("display.max_columns", 200)
PROJECT_ROOT = Path().resolve().parents[1]

## Load Dataset

In [2]:
df, feature_cols = load_gnn_ml_dataset(PROJECT_ROOT)
df.shape, len(feature_cols)

((145536, 69), 64)

In [3]:
trainer = MLTrainAndStore(
    df=df,
    feature_cols=feature_cols,
    target_col="systemic_risk_label",
)

trainer.train_df.shape, trainer.val_df.shape, trainer.test_df.shape

((109152, 69), (18192, 69), (13644, 69))

## Define Models

In [4]:
candidate_models = {
    "linear_regression": make_log_regression_model(LinearRegression(), scale_features=True),
}

list(candidate_models)

['linear_regression']

## Train And Store

In [5]:
trainer.train_many(candidate_models)

,model,train_mae,validation_mae,test_mae,train_rmse,validation_rmse,test_rmse,train_r2,validation_r2,test_r2
0,linear_regression,0.1802,0.351474,0.220549,1.684214,3.089513,1.722954,0.184947,-0.026151,0.242048


In [6]:
trainer.results()

,model,train_mae,validation_mae,test_mae,train_rmse,validation_rmse,test_rmse,train_r2,validation_r2,test_r2
0,linear_regression,0.1802,0.351474,0.220549,1.684214,3.089513,1.722954,0.184947,-0.026151,0.242048


## Single-Model Pattern

In [7]:
amodel = make_log_regression_model(
    XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42),
    scale_features=True,
)

trainer.train_and_store(model=amodel, name="XGBRegressor_search_1")
trainer.results()

,model,train_mae,validation_mae,test_mae,train_rmse,validation_rmse,test_rmse,train_r2,validation_r2,test_r2
0,XGBRegressor_search_1,0.056565,0.242722,0.165863,0.767725,2.989629,1.944634,0.830643,0.039128,0.03446
1,linear_regression,0.1802,0.351474,0.220549,1.684214,3.089513,1.722954,0.184947,-0.026151,0.242048


## Best Model

In [8]:
trainer.best_model_name()

'XGBRegressor_search_1'

In [9]:
trainer.predict_test().head(20)

,bank_id,year,quarter,period,systemic_risk_label,prediction,abs_error
0,5,2023,1,2023Q1,72,2.213876,69.786124
1,5,2023,3,2023Q3,65,0.914092,64.085908
2,5,2023,2,2023Q2,60,1.679206,58.320794
3,0,2023,1,2023Q1,56,2.522843,53.477157
4,0,2023,3,2023Q3,53,1.007473,51.992527
5,8,2023,1,2023Q1,53,1.837166,51.162834
6,0,2023,2,2023Q2,45,2.315583,42.684417
7,17,2023,1,2023Q1,44,1.935891,42.064109
8,6,2023,1,2023Q1,42,1.916627,40.083373
9,1,2023,1,2023Q1,41,2.268597,38.731403
